In [3]:
"""
Hello World: USDA Cropland Data Layer (CDL)
============================================
Annual 30m raster of crop types across the continental US.
No authentication required.

IMPORTANT — Coordinate system:
  The CDL API expects bounding box coordinates in the CONUS Albers Equal Area
  projection (EPSG:5070), NOT WGS84 lat/lon. Passing lat/lon directly causes
  a 500 server error. This script reprojects your WGS84 bbox automatically
  using pyproj before sending the request.

Access pattern:
  1. Reproject WGS84 bbox → EPSG:5070 (Albers)
  2. GET request to CropScape with reprojected bbox
  3. Parse XML response to extract GeoTIFF download URL
  4. Download and read GeoTIFF with rasterio

Each pixel is an integer crop code. Common codes:
  0   Background / nodata     1   Corn
  2   Cotton                  3   Rice
  4   Sorghum                 5   Soybeans
  21  Barley                  24  Winter Wheat
  36  Alfalfa                 111 Open Water
  190 Woody Wetlands

Full code list:
  https://www.nass.usda.gov/Research_and_Science/Cropland/docs/cdl_codes.pdf

API docs: https://nassgeodata.gmu.edu/CropScape/

Dependencies:
    pip install requests rasterio numpy pyproj
"""

import requests
import numpy as np
import xml.etree.ElementTree as ET
from io import BytesIO
from pyproj import Transformer

try:
    import rasterio
except ImportError:
    raise ImportError(
        "rasterio is required for CDL raster parsing.\n"
        "Install with: pip install rasterio"
    )

# Common crop code lookup table
CDL_CODES = {
    0:   "Background",
    1:   "Corn",
    2:   "Cotton",
    3:   "Rice",
    4:   "Sorghum",
    5:   "Soybeans",
    6:   "Sunflower",
    21:  "Barley",
    22:  "Durum Wheat",
    23:  "Spring Wheat",
    24:  "Winter Wheat",
    36:  "Alfalfa",
    37:  "Other Hay/Non Alfalfa",
    111: "Open Water",
    121: "Developed/Open Space",
    141: "Deciduous Forest",
    190: "Woody Wetlands",
    195: "Herbaceous Wetlands",
}


def wgs84_to_albers(min_lon, min_lat, max_lon, max_lat):
    """
    Reproject a WGS84 bounding box to CONUS Albers (EPSG:5070).

    The CDL API requires coordinates in Albers projection — passing WGS84
    lat/lon directly will cause a 500 server error.

    Returns:
        (min_x, min_y, max_x, max_y) in Albers metres
    """
    transformer = Transformer.from_crs("epsg:4326", "epsg:5070", always_xy=True)
    min_x, min_y = transformer.transform(min_lon, min_lat)
    max_x, max_y = transformer.transform(max_lon, max_lat)
    return min_x, min_y, max_x, max_y


def fetch_cdl(year=2022, bbox_wgs84=(-91.2, 34.8, -91.0, 35.0)):
    """
    Download a CDL GeoTIFF for a WGS84 bounding box and return pixel array.

    Args:
        year:       Calendar year (CDL available from 2008 onward for most states)
        bbox_wgs84: (min_lon, min_lat, max_lon, max_lat) in WGS84 decimal degrees

    Returns:
        2D numpy array of crop codes, or None on failure.
    """
    min_lon, min_lat, max_lon, max_lat = bbox_wgs84

    # Reproject to Albers — required by the CDL API
    min_x, min_y, max_x, max_y = wgs84_to_albers(min_lon, min_lat, max_lon, max_lat)
    albers_bbox = f"{min_x:.0f},{min_y:.0f},{max_x:.0f},{max_y:.0f}"

    url = (
        "https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLFile"
        f"?year={year}&bbox={albers_bbox}"
    )

    print("── USDA Cropland Data Layer (CDL) ──────────────────────")
    print(f"Year:        {year}")
    print(f"BBox WGS84:  {bbox_wgs84}  (min_lon, min_lat, max_lon, max_lat)")
    print(f"BBox Albers: {albers_bbox}  (EPSG:5070 — required by CDL API)")
    print(f"URL:         {url}\n")

    # Step 1: request XML response containing GeoTIFF download URL
    resp = requests.get(url, timeout=60)

    if resp.status_code == 500:
        print("500 Server Error — common causes:")
        print("  - bbox outside continental US")
        print("  - CDL not available for this year/region")
        print(f"Raw response: {resp.text[:300]}")
        return None

    resp.raise_for_status()

    root = ET.fromstring(resp.content)

    # The download URL is in a <returnURL> element (namespace-agnostic search)
    url_elem = root.find(".//{*}returnURL") or root.find(".//returnURL")
    if url_elem is None or not url_elem.text:
        print("ERROR: Could not parse GeoTIFF URL from CDL response.")
        print(f"Raw response: {resp.text[:400]}")
        return None

    tiff_url = url_elem.text.strip()
    print(f"GeoTIFF URL: {tiff_url}\n")

    # Step 2: download GeoTIFF into memory
    tiff_resp = requests.get(tiff_url, timeout=60)
    tiff_resp.raise_for_status()

    # Step 3: open with rasterio and extract pixel array
    with rasterio.open(BytesIO(tiff_resp.content)) as src:
        data = src.read(1)   # single band — one crop code per pixel
        print(f"Raster shape: {data.shape[0]} rows × {data.shape[1]} cols")
        print(f"Resolution:   30m pixels")
        print(f"CRS:          {src.crs}\n")

    return data


def summarise(data, top_n=8):
    """Print a crop coverage summary table for a CDL pixel array."""
    codes, counts = np.unique(data, return_counts=True)
    total = counts.sum()

    print(f"{'Crop':<25} {'Code':>5} {'Pixels':>8} {'Coverage':>10}")
    print("-" * 52)
    for code, count in sorted(zip(codes, counts), key=lambda x: -x[1])[:top_n]:
        label = CDL_CODES.get(int(code), f"Code {code}")
        pct = count / total * 100
        print(f"{label:<25} {int(code):>5} {count:>8,} {pct:>9.1f}%")


if __name__ == "__main__":
    # Eastern Arkansas rice belt
    data = fetch_cdl(
        year=2022,
        bbox_wgs84=(-91.2, 34.8, -91.0, 35.0),
    )
    print("Isaac print")
    print(data.shape)
    print(data)
    if data is not None:
        summarise(data)

    print("\n── Done ────────────────────────────────────────────────")

── USDA Cropland Data Layer (CDL) ──────────────────────
Year:        2022
BBox WGS84:  (-91.2, 34.8, -91.0, 35.0)  (min_lon, min_lat, max_lon, max_lat)
BBox Albers: 435427,1315425,452377,1338704  (EPSG:5070 — required by CDL API)
URL:         https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLFile?year=2022&bbox=435427,1315425,452377,1338704



/var/folders/xh/br51g1_j71sdmy2fqdss_blh0000gn/T/ipykernel_13958/350862241.py:132: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  url_elem = root.find(".//{*}returnURL") or root.find(".//returnURL")


GeoTIFF URL: https://nassgeodata.gmu.edu/webservice/nass_data_cache/CDL_2022_clip_20260611004332_47001741.tif

Raster shape: 776 rows × 566 cols
Resolution:   30m pixels
CRS:          EPSG:5070

Isaac print
(776, 566)
[[  0 121  61 ...   5   5   5]
 [  0  61   5 ...   5   5   5]
 [  0  61  37 ...   5   5   5]
 ...
 [  0   3   3 ...   5   5   5]
 [  0   3   3 ...   5   5   5]
 [  0   3   3 ...   5   5   5]]
Crop                       Code   Pixels   Coverage
----------------------------------------------------
Soybeans                      5  194,725      44.3%
Rice                          3   75,312      17.1%
Woody Wetlands              190   62,529      14.2%
Corn                          1   47,973      10.9%
Code 61                      61   22,188       5.1%
Developed/Open Space        121    8,750       2.0%
Code 122                    122    6,761       1.5%
Cotton                        2    4,721       1.1%

── Done ────────────────────────────────────────────────


In [11]:
data = fetch_cdl(year=2025, bbox_wgs84=(-93.028385, 42.556797, -92.791956, 42.743418))
summarise(data)

── USDA Cropland Data Layer (CDL) ──────────────────────
Year:        2025
BBox WGS84:  (-93.028385, 42.556797, -92.791956, 42.743418)  (min_lon, min_lat, max_lon, max_lat)
BBox Albers: 242477,2176967,261058,2198440  (EPSG:5070 — required by CDL API)
URL:         https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLFile?year=2025&bbox=242477,2176967,261058,2198440



/var/folders/xh/br51g1_j71sdmy2fqdss_blh0000gn/T/ipykernel_13958/350862241.py:132: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  url_elem = root.find(".//{*}returnURL") or root.find(".//returnURL")


GeoTIFF URL: https://nassgeodata.gmu.edu/webservice/nass_data_cache/CDL_2025_clip_20260611005056_1472519863.tif

Raster shape: 716 rows × 620 cols
Resolution:   30m pixels
CRS:          EPSG:5070

Crop                       Code   Pixels   Coverage
----------------------------------------------------
Corn                          1  176,248      39.7%
Soybeans                      5  123,333      27.8%
Code 176                    176   68,815      15.5%
Code 122                    122   13,460       3.0%
Herbaceous Wetlands         195   12,629       2.8%
Woody Wetlands              190   12,245       2.8%
Other Hay/Non Alfalfa        37    7,903       1.8%
Developed/Open Space        121    7,173       1.6%
